# Sistema de Verificação de Decolagem - Aurora-Singer

Este projeto simula um **sistema de análise de telemetria** usado antes da decolagem de uma nave espacial.

O objetivo é analisar dados de sensores e verificar se todos os parâmetros estarão dentro das faixas seguras para permitir a decolagem.

Caso algum valor esteja fora do limite seguro, o sistema aborta automaticamente a missão.

A decisão final é feita por **regras fixas**, enquanto a **IA atua como apoio** na estimativa de risco e detecção de anomalias.

## 1.1 Organização da Telemetria

Os dados de telemetria são coletados pelos sensores da nave e analisados antes da decolagem.
O sistema verifica se todos os valores estão dentro das faixas seguras para evitar riscos durante o lançamento.

| Parâmetro | Descrição | Faixa Segura | Unidade |
|-----------|-----------|--------------|----------|
| Temperatura Interna | Temperatura dentro da nave | 20 - 32 | °C |
| Temperatura Externa | Temperatura ambiente | 0 - 28 | °C |
| Tensão da Bateria | Voltagem do sistema | 47 - 51.5 | V |
| Corrente da Bateria | Corrente de saída | 25 - 100 | A |
| Nível de Energia | Estado de carga da bateria | ≥ 70 | % |
| Pressão dos Tanques | Pressão do combustível | 100 - 140 | bar |
| Integridade Estrutural | Estado da estrutura | = 1 (Íntegra) | Status |
| Módulos Críticos | Funcionamento dos sistemas | = 1 (OK) | Status |
| Link de Telemetria | Comunicação com centro | = 1 (Ativo) | Status |
| Perdas Energéticas | Perdas no sistema | ≤ 7 | % |

## 1.2 Algoritmo de Verificação

### Pseudocódigo

```
INÍCIO

  LER dados de telemetria dos sensores
  
  SE temperatura_interna NÃO está entre 20°C e 32°C
    ABORTAR com motivo "Temperatura interna fora da faixa"
  FIM SE
  
  SE temperatura_externa NÃO está entre 0°C e 28°C
    ABORTAR com motivo "Temperatura externa fora da faixa"
  FIM SE
  
  SE tensao_bateria NÃO está entre 47V e 51.5V
    ABORTAR com motivo "Tensão da bateria fora da faixa"
  FIM SE
  
  SE corrente_bateria NÃO está entre 25A e 100A
    ABORTAR com motivo "Corrente da bateria fora da faixa"
  FIM SE
  
  SE nivel_energia < 70%
    ABORTAR com motivo "Nível de energia insuficiente"
  FIM SE
  
  SE perdas_energeticas > 7%
    ABORTAR com motivo "Perdas energéticas acima do limite"
  FIM SE
  
  SE pressao_tanques NÃO está entre 100 bar e 140 bar
    ABORTAR com motivo "Pressão dos tanques fora da faixa"
  FIM SE
  
  SE integridade_estrutural ≠ 1
    ABORTAR com motivo "Integridade estrutural comprometida"
  FIM SE
  
  SE status_modulos_criticos ≠ 1
    ABORTAR com motivo "Falha nos módulos críticos"
  FIM SE
  
  SE status_link_telemetria ≠ 1
    ABORTAR com motivo "Falha no link de telemetria"
  FIM SE
  
  CASO CONTRÁRIO
    RETORNAR "PRONTO PARA DECOLAR"
  FIM CASO

FIM
```

## Bibliotecas Utilizadas

- **`pandas`**: para ler e organizar o dataset sintético em formato CSV
- **`DecisionTreeClassifier`**: para estimar o nível de risco baseado em dados históricos
- **`IsolationForest`**: para detectar possíveis anomalias nos padrões de telemetria

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest

## Carregamento de Dados

Utilizamos um **dataset sintético de telemetria** armazenado em `telemetria_sintetica.csv`.
O dataset contém 100 registros com leitura de sensores de uma nave espacial.

In [ ]:
CAMINHO_CSV = "telemetria_sintetica.csv"
df = pd.read_csv(CAMINHO_CSV)
print(f"Dataset carregado com {len(df)} registros")
display(df.head(10))

## 1.3 Funções Principais

### Transformação de Dados
Mapeamento de colunas CSV para estrutura de telemetria.

### Verificação de Decolagem
Implementa as 10 verificações sequenciais do algoritmo.

### Classificação de Risco
Mapeia confiança do modelo para níveis de risco.

### Análise Energética
Calcula autonomia e consumo de energia.

In [ ]:
def carregar_dataset(caminho_csv):
    return pd.read_csv(caminho_csv)


def linha_para_telemetria(row):
    """Converte linha do CSV para dicionário de telemetria"""
    return {
        "timestamp": row["timestamp"],
        "temperatura_interna": row["internal_temp_c"],
        "temperatura_externa": row["external_temp_c"],
        "tensao_bateria": row["battery_voltage_v"],
        "corrente_bateria": row["battery_current_a"],
        "nivel_energia": row["battery_soc_percent"],
        "capacidade_bateria_ah": row["battery_capacity_ah"],
        "energia_disponivel_kwh": row["energy_available_kwh"],
        "carga_kw": row["power_load_kw"],
        "perdas_energeticas": row["energy_loss_percent"],
        "pressao_tanques": row["tank_pressure_bar"],
        "integridade_estrutural": row["structural_integrity"],
        "status_modulos_criticos": row["critical_modules_status"],
        "status_link_telemetria": row["telemetry_link_status"],
        "autonomia_estimada_dataset_min": row["estimated_autonomy_min"],
        "decisao_dataset": row["launch_decision"],
    }


def verificar_decolagem(dados):
    """Executa 10 verificações de segurança para liberação da decolagem"""
    falhas = []

    if not (20 <= dados["temperatura_interna"] <= 32):
        falhas.append("Temperatura interna fora da faixa segura.")

    if not (0 <= dados["temperatura_externa"] <= 28):
        falhas.append("Temperatura externa fora da faixa segura.")

    if not (47 <= dados["tensao_bateria"] <= 51.5):
        falhas.append("Tensão da bateria fora da faixa segura.")

    if not (25 <= dados["corrente_bateria"] <= 100):
        falhas.append("Corrente da bateria fora da faixa segura.")

    if dados["nivel_energia"] < 70:
        falhas.append("Nível de energia insuficiente.")

    if dados["perdas_energeticas"] > 7:
        falhas.append("Perdas energéticas acima do limite.")

    if not (100 <= dados["pressao_tanques"] <= 140):
        falhas.append("Pressão dos tanques fora da faixa segura.")

    if dados["integridade_estrutural"] != 1:
        falhas.append("Integridade estrutural comprometida.")

    if dados["status_modulos_criticos"] != 1:
        falhas.append("Falha nos módulos críticos.")

    if dados["status_link_telemetria"] != 1:
        falhas.append("Falha no link de telemetria.")

    if len(falhas) == 0:
        return "PRONTO PARA DECOLAR", falhas

    return "DECOLAGEM ABORTADA", falhas


def classificar_risco_por_probabilidade(prob_ready):
    """Classifica risco em 3 níveis baseado em confiança do modelo"""
    if prob_ready >= 0.80:
        return "RISCO_BAIXO"
    elif prob_ready >= 0.50:
        return "RISCO_MODERADO"
    return "RISCO_ALTO"


def calcular_analise_energetica(energia_disponivel_kwh, perdas_percentual, carga_kw):
    """Calcula autonomia e consumo energético da decolagem"""
    energia_util = energia_disponivel_kwh * (1 - perdas_percentual / 100)
    consumo_estimado_decolagem_kwh = carga_kw / 60
    energia_restante_apos_decolagem = max(energia_util - consumo_estimado_decolagem_kwh, 0)
    autonomia_estimada_min = (energia_util / carga_kw) * 60 if carga_kw > 0 else 0

    return (
        energia_util,
        consumo_estimado_decolagem_kwh,
        energia_restante_apos_decolagem,
        autonomia_estimada_min,
    )

print("✓ Funções definidas com sucesso")

## 1.4 Treinamento dos Modelos de IA

Treinamos dois modelos com base no dataset histórico:
- **DecisionTree**: Prediz se a decolagem será READY ou ABORT
- **IsolationForest**: Detecta anomalias nos padrões de telemetria

In [ ]:
colunas_features = [
    "internal_temp_c",
    "external_temp_c",
    "battery_voltage_v",
    "battery_current_a",
    "battery_soc_percent",
    "energy_available_kwh",
    "power_load_kw",
    "energy_loss_percent",
    "tank_pressure_bar",
    "estimated_autonomy_min",
]

# Treinamento DecisionTree
modelo_decisao = DecisionTreeClassifier(random_state=42)
modelo_decisao.fit(df[colunas_features], df["launch_decision"])

# Treinamento IsolationForest
modelo_anomalia = IsolationForest(random_state=42, contamination=0.1)
modelo_anomalia.fit(df[colunas_features])

print("✓ Modelos treinados com sucesso")
print(f"  - DecisionTree com {len(colunas_features)} features")
print(f"  - IsolationForest com contamination=0.1")

## Simulação da Telemetria Atual

Selecionamos uma amostra do dataset como se fosse a leitura atual dos sensores.

In [ ]:
# Selecionamos a primeira linha com decisão READY (sucesso)
indices_ready = df.index[df["launch_decision"] == "READY"].tolist()
INDICE_AMOSTRA = indices_ready[0] if indices_ready else 0

row_atual = df.iloc[INDICE_AMOSTRA]
telemetria = linha_para_telemetria(row_atual)
features_atual = df.loc[[INDICE_AMOSTRA], colunas_features]

print(f"Amostra selecionada: índice {INDICE_AMOSTRA}")
print(f"Timestamp: {telemetria['timestamp']}")
print(f"Decisão esperada no dataset: {telemetria['decisao_dataset']}")

## Execução da Verificação de Decolagem

O sistema executa as 10 regras de segurança do algoritmo.

In [ ]:
status, falhas = verificar_decolagem(telemetria)

print("="*70)
print("RELATÓRIO DE VERIFICAÇÃO - AURORA-SINGER")
print("="*70)
print(f"\n🚀 DECISÃO FINAL: {status}\n")

if falhas:
    print("Motivos da verificação:")
    for i, falha in enumerate(falhas, 1):
        print(f"  {i}. ❌ {falha}")
else:
    print("✅ Todos os parâmetros dentro dos limites seguros!")

print("\n" + "="*70)

## 1.5 Análise de Risco

O sistema classifica o nível de risco baseado em múltiplos fatores.

In [ ]:
print("\n📊 ANÁLISE DE RISCO\n")

# Verifica níveis críticos
if telemetria["nivel_energia"] < 60:
    classificacao_risco = "RISCO ALTO"
elif telemetria["nivel_energia"] < 75:
    classificacao_risco = "RISCO MODERADO"
else:
    classificacao_risco = "RISCO BAIXO"

print(f"Classificação: {classificacao_risco}")
print(f"  • Nível de Energia: {telemetria['nivel_energia']:.1f}%")
print(f"  • Temperatura Interna: {telemetria['temperatura_interna']:.1f}°C")
print(f"  • Pressão dos Tanques: {telemetria['pressao_tanques']:.1f} bar")
print(f"  • Integridade Estrutural: OK")

## 1.6 Análise Energética

Cálculo detalhado de autonomia e consumo de energia.

In [ ]:
energia_util, consumo_estimado_decolagem_kwh, energia_restante_apos_decolagem, autonomia_estimada_min = calcular_analise_energetica(
    telemetria["energia_disponivel_kwh"],
    telemetria["perdas_energeticas"],
    telemetria["carga_kw"],
)

print("\n⚡ ANÁLISE ENERGÉTICA\n")
print(f"Energia disponível (bruto): {telemetria['energia_disponivel_kwh']:.3f} kWh")
print(f"Perdas energéticas: {telemetria['perdas_energeticas']:.1f}%")
print(f"Energia útil (após perdas): {energia_util:.3f} kWh")
print(f"\nConsumo estimado (1 min decolagem): {consumo_estimado_decolagem_kwh:.3f} kWh")
print(f"Carga atual: {telemetria['carga_kw']:.1f} kW")
print(f"\nEnergia restante após decolagem: {energia_restante_apos_decolagem:.3f} kWh")
print(f"Autonomia estimada: {autonomia_estimada_min:.2f} minutos")
print(f"Autonomia registrada no dataset: {telemetria['autonomia_estimada_dataset_min']} minutos")

## 1.7 Análise Assistida por IA

Machine Learning como ferramenta de apoio para estimativa de risco e detecção de anomalias.

In [ ]:
# Predição do modelo Decision Tree
probs = modelo_decisao.predict_proba(features_atual)[0]
classes = list(modelo_decisao.classes_)

if "READY" in classes:
    prob_ready = probs[classes.index("READY")]
else:
    prob_ready = 0.0

risco_ia = classificar_risco_por_probabilidade(prob_ready)

# Detecção de anomalias
anomalia = modelo_anomalia.predict(features_atual)[0]
mensagem_anomalia = "ANOMALIA DETECTADA ⚠️" if anomalia == -1 else "Padrão normal ✓"

print("\n🤖 ANÁLISE DE INTELIGÊNCIA ARTIFICIAL\n")
print(f"Confiança do modelo (prob. READY): {prob_ready:.1%}")
print(f"Risco estimado pela IA: {risco_ia}")
print(f"\nDetecção de Anomalias: {mensagem_anomalia}")
print(f"\n💡 A IA atua como ferramenta de apoio.")
print(f"   A decisão final é sempre baseada nas REGRAS DE SEGURANÇA.")

## 1.8 Reflexão Crítica

Análise dos aspectos éticos, sociais e de sustentabilidade da exploração espacial.

### Ética na Exploração Espacial

A responsabilidade das agências espaciais vai além do aspecto técnico. Ela envolve um compromisso com a preservação da vida humana e com o futuro da exploração espacial.

**1. Dever de Cuidado (Preservação da Vida)**

A segurança deve ser tratada como prioridade máxima. As agências têm a responsabilidade de evitar expor astronautas ou populações civis a riscos desnecessários em nome do progresso tecnológico ou econômico. Sistemas como Aurora-Singer exemplificam esse compromisso ao implementar múltiplas camadas de verificação antes de qualquer operação crítica.

**2. Gestão de Bens Comuns (Sustentabilidade do Espaço)**

O espaço é considerado um patrimônio da humanidade. Por isso, é importante que as missões sejam planejadas de forma a reduzir a geração de lixo espacial, preservando o ambiente orbital para futuras gerações. A síndrome de Kessler - uma reação em cadeia de colisões - deve ser evitada a todo custo.

**3. Transparência e Responsabilidade**

As organizações envolvidas devem comunicar de forma clara os riscos das missões e assumir responsabilidade por eventuais danos causados, seja no espaço ou na Terra. A documentação de sistemas como este notebook demonstra transparência nos processos.

**4. Justiça Global**

A exploração espacial deve gerar benefícios para toda a humanidade, e não apenas para países mais ricos. Além disso, falhas de segurança podem afetar infraestruturas globais, como satélites de comunicação e monitoramento climático.

**Conclusão Ética:** A ética na exploração espacial exige que as atividades sejam conduzidas de maneira segura, sustentável e responsável, tratando o espaço como um ambiente que requer cooperação internacional e respeito à vida.

### Impacto Social

O impacto social de segurança nas missões espaciais está diretamente ligado à confiança da sociedade e à continuidade da exploração tecnológica.

**1. Confiança e Apoio Público**

O sucesso das missões fortalece o interesse da sociedade e o apoio ao investimento público em ciência e tecnologia. Por outro lado, acidentes históricos (como o desastre do Challenger) geram forte impacto emocional e podem levar à interrupção de programas espaciais por longos períodos.

**2. Democratização de Acesso ao Espaço**

Com padrões de segurança cada vez mais rigorosos, a exploração espacial deixa de ser exclusiva de militares e cientistas, abrindo espaço para a participação de civis, pesquisadores e até turismo espacial. Sistemas robustos como Aurora-Singer permitem operações mais seguras e confiáveis.

**3. Proteção de Infraestruturas Essenciais**

A segurança nas operações espaciais também protege satélites fundamentais para a sociedade moderna, responsáveis por serviços como GPS, comunicação, previsão do tempo e monitoramento de desastres naturais. Uma falha em uma operação pode ter efeito cascata na infraestrutura global.

**4. Inspiração e Educação**

Missões espaciais bem-sucedidas e seguras têm forte impacto educacional, inspirando jovens a seguir carreiras nas áreas de ciência, tecnologia, engenharia e matemática (STEM), contribuindo para o desenvolvimento científico e econômico.

**Conclusão Social:** A segurança nas missões espaciais transforma a exploração do espaço em um fator de desenvolvimento científico, tecnológico e social para a humanidade.

### Sustentabilidade

No contexto da exploração espacial, a sustentabilidade é um dos pilares fundamentais para garantir que o espaço continue sendo utilizado pelas próximas gerações.

**1. Preservação das Órbitas**

Uma das maiores preocupações é evitar a chamada Síndrome de Kessler - uma reação em cadeia onde colisões entre objetos geram grandes quantidades de detritos espaciais, podendo tornar determinadas órbitas inutilizáveis por longos períodos. Por isso, manter o espaço orbital livre de lixo é uma responsabilidade importante das agências espaciais.

**2. Gestão do Ciclo de Vida de Satélites**

Missões sustentáveis planejam o destino final dos equipamentos espaciais. Isso pode incluir a reentrada controlada na atmosfera ou o envio para chamadas "órbitas cemitério", evitando que satélites desativados se tornem detritos perigosos.

**3. Proteção Planetária**

Existe a preocupação de não contaminar outros corpos celestes, como Marte ou a Lua, com microrganismos da Terra. Essa medida preserva a integridade científica desses ambientes para futuras pesquisas.

**4. Eficiência no Uso de Recursos**

O desenvolvimento de tecnologia reutilizável, como foguetes capazes de pousar e ser utilizado novamente, contribui para reduzir desperdícios e diminuir o impacto ambiental dos lançamentos.

**Conclusão Sustentável:** A sustentabilidade espacial busca garantir que a exploração do espaço seja realizada de forma responsável, preservando o ambiente orbital e permitindo que futuras gerações também possam utilizá-los.

### Conclusão Geral

Em conjunto, os aspectos éticos, sociais e de sustentabilidade mostram que a exploração espacial não é apenas um desafio tecnológico, mas também uma responsabilidade coletiva da humanidade.

Garantir segurança nas missões, preservar o ambiente espacial e promover benefícios sociais amplos são fatores essenciais para que a exploração do espaço aconteça de forma responsável.

Dessa forma, o avanço científico pode ocorrer sem comprometer a segurança das pessoas, a preservação do ambiente orbital e o acesso das futuras gerações ao espaço.

**Sistemas como Aurora-Singer representam esse compromisso: tecnologia a serviço da segurança, da responsabilidade e da sustentabilidade.**

# Relatório de Pré-Decolagem AuroraSinger com Apoio de IA

Este notebook implementa:

- verificação de segurança por **regras fixas**;
- apoio da **IA** com estimativa de risco;
- detecção de **anomalias**;
- análise energética da decolagem.

A decisão final de **PRONTO PARA DECOLAR** ou **DECOLAGEM ABORTADA** é feita pelas regras de segurança.  
A IA atua apenas como análise de apoio.

## Bibliotecas utilizadas

Neste projeto, utilizamos:

- **`pandas`**: para ler e organizar o dataset sintético em formato CSV;
- **`DecisionTreeClassifier`**: para estimar o nível de risco da operação;
- **`IsolationForest`**: para detectar possíveis anomalias nos dados.

As bibliotecas foram escolhidas por permitirem integrar análise de dados e inteligência artificial de forma prática e organizada.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest

## Dados sintéticos utilizados

Neste projeto, utilizamos um **dataset sintético de telemetria** armazenado no arquivo `telemetria_sintetica.csv`, que foi gerado por Inteligência Artificial.

Abaixo, apresentamos uma prévia das primeiras linhas do dataset utilizado no experimento.

In [ ]:
CAMINHO_CSV = "telemetria_sintetica.csv"
df = pd.read_csv(CAMINHO_CSV)
display(df.head())

## Funções principais

Neste bloco, definimos:

- a conversão de uma linha do dataset para o formato de telemetria;
- a verificação da decolagem por regras;
- a classificação do risco com apoio da IA;
- o cálculo da análise energética.

In [ ]:
def carregar_dataset(caminho_csv):
    return pd.read_csv(caminho_csv)


def linha_para_telemetria(row):
    return {
        "timestamp": row["timestamp"],
        "temperatura_interna": row["internal_temp_c"],
        "temperatura_externa": row["external_temp_c"],
        "tensao_bateria": row["battery_voltage_v"],
        "corrente_bateria": row["battery_current_a"],
        "nivel_energia": row["battery_soc_percent"],
        "capacidade_bateria_ah": row["battery_capacity_ah"],
        "energia_disponivel_kwh": row["energy_available_kwh"],
        "carga_kw": row["power_load_kw"],
        "perdas_energeticas": row["energy_loss_percent"],
        "pressao_tanques": row["tank_pressure_bar"],
        "integridade_estrutural": row["structural_integrity"],
        "status_modulos_criticos": row["critical_modules_status"],
        "status_link_telemetria": row["telemetry_link_status"],
        "autonomia_estimada_dataset_min": row["estimated_autonomy_min"],
        "decisao_dataset": row["launch_decision"],
    }


def verificar_decolagem(dados):
    falhas = []

    if not (20 <= dados["temperatura_interna"] <= 32):
        falhas.append("Temperatura interna fora da faixa segura.")

    if not (0 <= dados["temperatura_externa"] <= 28):
        falhas.append("Temperatura externa fora da faixa segura.")

    if not (47 <= dados["tensao_bateria"] <= 51.5):
        falhas.append("Tensão da bateria fora da faixa segura.")

    if not (25 <= dados["corrente_bateria"] <= 100):
        falhas.append("Corrente da bateria fora da faixa segura.")

    if dados["nivel_energia"] < 70:
        falhas.append("Nível de energia insuficiente.")

    if dados["perdas_energeticas"] > 7:
        falhas.append("Perdas energéticas acima do limite.")

    if not (100 <= dados["pressao_tanques"] <= 140):
        falhas.append("Pressão dos tanques fora da faixa segura.")

    if dados["integridade_estrutural"] != 1:
        falhas.append("Integridade estrutural comprometida.")

    if dados["status_modulos_criticos"] != 1:
        falhas.append("Falha nos módulos críticos.")

    if dados["status_link_telemetria"] != 1:
        falhas.append("Falha no link de telemetria.")

    if len(falhas) == 0:
        return "PRONTO PARA DECOLAR", falhas

    return "DECOLAGEM ABORTADA", falhas


def classificar_risco_por_probabilidade(prob_ready):
    if prob_ready >= 0.80:
        return "RISCO_BAIXO"
    elif prob_ready >= 0.50:
        return "RISCO_MODERADO"
    return "RISCO_ALTO"


def calcular_analise_energetica(energia_disponivel_kwh, perdas_percentual, carga_kw):
    energia_util = energia_disponivel_kwh * (1 - perdas_percentual / 100)

    consumo_estimado_decolagem_kwh = carga_kw / 60

    energia_restante_apos_decolagem = max(
        energia_util - consumo_estimado_decolagem_kwh, 0
    )

    autonomia_estimada_min = (energia_util / carga_kw) * 60 if carga_kw > 0 else 0

    return (
        energia_util,
        consumo_estimado_decolagem_kwh,
        energia_restante_apos_decolagem,
        autonomia_estimada_min,
    )

## Treinamento dos modelos de IA

Aqui, carregamos o dataset sintético e treinamos:

- um modelo para **estimativa de risco**, com base na decisão `READY` ou `ABORT`;
- um modelo para **detecção de anomalias**.

A IA não toma a decisão final da decolagem.  
Ela complementa a análise feita pelas regras do sistema.

In [ ]:
CAMINHO_CSV = "telemetria_sintetica.csv"

df = carregar_dataset(CAMINHO_CSV)

colunas_features = [
    "internal_temp_c",
    "external_temp_c",
    "battery_voltage_v",
    "battery_current_a",
    "battery_soc_percent",
    "energy_available_kwh",
    "power_load_kw",
    "energy_loss_percent",
    "tank_pressure_bar",
    "estimated_autonomy_min",
]

modelo_decisao = DecisionTreeClassifier(random_state=42)
modelo_decisao.fit(df[colunas_features], df["launch_decision"])

modelo_anomalia = IsolationForest(random_state=42, contamination=0.1)
modelo_anomalia.fit(df[colunas_features])

## Simulação da telemetria

Nessa etapa, utilizamos uma linha do próprio dataset como se fosse a leitura atual dos sensores da nave.

Por padrão, o notebook seleciona a primeira linha cuja decisão no dataset seja `READY`.  
Caso não exista nenhuma, utiliza a primeira linha disponível.

In [ ]:
indices_ready = df.index[df["launch_decision"] == "READY"].tolist()
INDICE_AMOSTRA = indices_ready[0] if indices_ready else 0

row_atual = df.iloc[INDICE_AMOSTRA]
telemetria = linha_para_telemetria(row_atual)
features_atual = df.loc[[INDICE_AMOSTRA], colunas_features]

display(pd.DataFrame([telemetria]))

## Execução da verificação

Aqui o sistema:

1. avalia as regras de segurança;
2. verifica com a IA o nível de risco;
3. identifica se há anomalias;
4. imprime o relatório de pré-decolagem.

In [ ]:
status, falhas = verificar_decolagem(telemetria)

probs = modelo_decisao.predict_proba(features_atual)[0]
classes = list(modelo_decisao.classes_)

if "READY" in classes:
    prob_ready = probs[classes.index("READY")]
else:
    prob_ready = 0.0

risco_ia = classificar_risco_por_probabilidade(prob_ready)

anomalia = modelo_anomalia.predict(features_atual)[0]

if anomalia == -1:
    mensagem_ia = "comportamento anômalo detectado."
else:
    mensagem_ia = "nenhuma anomalia detectada."

print("=== RELATÓRIO DE PRÉ-DECOLAGEM ===")
for chave, valor in telemetria.items():
    print(f"{chave}: {valor}")

print("\nResultado final:", status)
print("Nível de risco estimado pela IA:", risco_ia)
print("IA:", mensagem_ia)

if falhas:
    print("Motivos da abortagem:")
    for falha in falhas:
        print("-", falha)

## Análise energética

Agora calculamos:

- energia disponível antes das perdas;
- energia útil após as perdas;
- consumo estimado da decolagem;
- energia restante após a decolagem;
- autonomia estimada.

Nesta etapa, os valores são obtidos a partir do próprio dataset sintético.

In [ ]:
energia_util, consumo_estimado_decolagem_kwh, energia_restante_apos_decolagem, autonomia_estimada_min = calcular_analise_energetica(
    telemetria["energia_disponivel_kwh"],
    telemetria["perdas_energeticas"],
    telemetria["carga_kw"],
)

print("=== ANÁLISE ENERGÉTICA ===")
print(f"Energia disponível antes das perdas: {telemetria['energia_disponivel_kwh']:.3f} kWh")
print(f"Energia útil após perdas: {energia_util:.3f} kWh")
print(f"Consumo estimado da decolagem (1 min): {consumo_estimado_decolagem_kwh:.3f} kWh")
print(f"Energia restante após decolagem: {energia_restante_apos_decolagem:.3f} kWh")
print(f"Autonomia estimada após perdas: {autonomia_estimada_min:.2f} min")
print(f"Autonomia registrada no dataset: {telemetria['autonomia_estimada_dataset_min']} min")

## Conclusão

O sistema combina duas abordagens:

- **regras fixas:** responsáveis pela decisão final de segurança;
- **IA:** utilizada como apoio para estimativa de risco e detecção de anomalias.

Essa separação continua sendo importante porque, em sistemas críticos, a decisão operacional deve ser baseada em critérios explícitos e auditáveis, enquanto a IA deve atuar como ferramenta complementar de análise.

# Relatório de Pré-Decolagem AuroraSinger com Apoio de IA

Este notebook implementa:

- verificação de segurança por **regras fixas**;
- apoio da **IA** com estimativa de risco;
- detecção de **anomalias**;
- análise energética da decolagem.

A decisão final de **PRONTO PARA DECOLAR** ou **DECOLAGEM ABORTADA** é feita pelas regras de segurança.  
A IA atua apenas como análise de apoio.

## Bibliotecas utilizadas

Neste projeto, utilizamos:

- **`pandas`**: para ler e organizar o dataset sintético em formato CSV;
- **`DecisionTreeClassifier`**: para estimar o nível de risco da operação;
- **`IsolationForest`**: para detectar possíveis anomalias nos dados.

As bibliotecas foram escolhidas por permitirem integrar análise de dados e inteligência artificial de forma prática e organizada.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest

## Dados sintéticos utilizados

Neste projeto, utilizamos um **dataset sintético de telemetria** armazenado no arquivo `telemetria_sintetica.csv`, que foi gerado por Inteligência Artificial.

Abaixo, apresentamos uma prévia das primeiras linhas do dataset utilizado no experimento.

In [ ]:
CAMINHO_CSV = "telemetria_sintetica.csv"
df = pd.read_csv(CAMINHO_CSV)
display(df.head())

## Funções principais

Neste bloco, definimos:

- a conversão de uma linha do dataset para o formato de telemetria;
- a verificação da decolagem por regras;
- a classificação do risco com apoio da IA;
- o cálculo da análise energética.

In [ ]:
def carregar_dataset(caminho_csv):
    return pd.read_csv(caminho_csv)


def linha_para_telemetria(row):
    return {
        "timestamp": row["timestamp"],
        "temperatura_interna": row["internal_temp_c"],
        "temperatura_externa": row["external_temp_c"],
        "tensao_bateria": row["battery_voltage_v"],
        "corrente_bateria": row["battery_current_a"],
        "nivel_energia": row["battery_soc_percent"],
        "capacidade_bateria_ah": row["battery_capacity_ah"],
        "energia_disponivel_kwh": row["energy_available_kwh"],
        "carga_kw": row["power_load_kw"],
        "perdas_energeticas": row["energy_loss_percent"],
        "pressao_tanques": row["tank_pressure_bar"],
        "integridade_estrutural": row["structural_integrity"],
        "status_modulos_criticos": row["critical_modules_status"],
        "status_link_telemetria": row["telemetry_link_status"],
        "autonomia_estimada_dataset_min": row["estimated_autonomy_min"],
        "decisao_dataset": row["launch_decision"],
    }


def verificar_decolagem(dados):
    falhas = []

    if not (20 <= dados["temperatura_interna"] <= 32):
        falhas.append("Temperatura interna fora da faixa segura.")

    if not (0 <= dados["temperatura_externa"] <= 28):
        falhas.append("Temperatura externa fora da faixa segura.")

    if not (47 <= dados["tensao_bateria"] <= 51.5):
        falhas.append("Tensão da bateria fora da faixa segura.")

    if not (25 <= dados["corrente_bateria"] <= 100):
        falhas.append("Corrente da bateria fora da faixa segura.")

    if dados["nivel_energia"] < 70:
        falhas.append("Nível de energia insuficiente.")

    if dados["perdas_energeticas"] > 7:
        falhas.append("Perdas energéticas acima do limite.")

    if not (100 <= dados["pressao_tanques"] <= 140):
        falhas.append("Pressão dos tanques fora da faixa segura.")

    if dados["integridade_estrutural"] != 1:
        falhas.append("Integridade estrutural comprometida.")

    if dados["status_modulos_criticos"] != 1:
        falhas.append("Falha nos módulos críticos.")

    if dados["status_link_telemetria"] != 1:
        falhas.append("Falha no link de telemetria.")

    if len(falhas) == 0:
        return "PRONTO PARA DECOLAR", falhas

    return "DECOLAGEM ABORTADA", falhas


def classificar_risco_por_probabilidade(prob_ready):
    if prob_ready >= 0.80:
        return "RISCO_BAIXO"
    elif prob_ready >= 0.50:
        return "RISCO_MODERADO"
    return "RISCO_ALTO"


def calcular_analise_energetica(energia_disponivel_kwh, perdas_percentual, carga_kw):
    energia_util = energia_disponivel_kwh * (1 - perdas_percentual / 100)

    consumo_estimado_decolagem_kwh = carga_kw / 60

    energia_restante_apos_decolagem = max(
        energia_util - consumo_estimado_decolagem_kwh, 0
    )

    autonomia_estimada_min = (energia_util / carga_kw) * 60 if carga_kw > 0 else 0

    return (
        energia_util,
        consumo_estimado_decolagem_kwh,
        energia_restante_apos_decolagem,
        autonomia_estimada_min,
    )

## Treinamento dos modelos de IA

Aqui, carregamos o dataset sintético e treinamos:

- um modelo para **estimativa de risco**, com base na decisão `READY` ou `ABORT`;
- um modelo para **detecção de anomalias**.

A IA não toma a decisão final da decolagem.  
Ela complementa a análise feita pelas regras do sistema.

In [ ]:
CAMINHO_CSV = "telemetria_sintetica.csv"

df = carregar_dataset(CAMINHO_CSV)

colunas_features = [
    "internal_temp_c",
    "external_temp_c",
    "battery_voltage_v",
    "battery_current_a",
    "battery_soc_percent",
    "energy_available_kwh",
    "power_load_kw",
    "energy_loss_percent",
    "tank_pressure_bar",
    "estimated_autonomy_min",
]

modelo_decisao = DecisionTreeClassifier(random_state=42)
modelo_decisao.fit(df[colunas_features], df["launch_decision"])

modelo_anomalia = IsolationForest(random_state=42, contamination=0.1)
modelo_anomalia.fit(df[colunas_features])

## Simulação da telemetria

Nessa etapa, utilizamos uma linha do próprio dataset como se fosse a leitura atual dos sensores da nave.

Por padrão, o notebook seleciona a primeira linha cuja decisão no dataset seja `READY`.  
Caso não exista nenhuma, utiliza a primeira linha disponível.

In [ ]:
indices_ready = df.index[df["launch_decision"] == "READY"].tolist()
INDICE_AMOSTRA = indices_ready[0] if indices_ready else 0

row_atual = df.iloc[INDICE_AMOSTRA]
telemetria = linha_para_telemetria(row_atual)
features_atual = df.loc[[INDICE_AMOSTRA], colunas_features]

display(pd.DataFrame([telemetria]))

## Execução da verificação

Aqui o sistema:

1. avalia as regras de segurança;
2. verifica com a IA o nível de risco;
3. identifica se há anomalias;
4. imprime o relatório de pré-decolagem.

In [ ]:
status, falhas = verificar_decolagem(telemetria)

probs = modelo_decisao.predict_proba(features_atual)[0]
classes = list(modelo_decisao.classes_)

if "READY" in classes:
    prob_ready = probs[classes.index("READY")]
else:
    prob_ready = 0.0

risco_ia = classificar_risco_por_probabilidade(prob_ready)

anomalia = modelo_anomalia.predict(features_atual)[0]

if anomalia == -1:
    mensagem_ia = "comportamento anômalo detectado."
else:
    mensagem_ia = "nenhuma anomalia detectada."

print("=== RELATÓRIO DE PRÉ-DECOLAGEM ===")
for chave, valor in telemetria.items():
    print(f"{chave}: {valor}")

print("\nResultado final:", status)
print("Nível de risco estimado pela IA:", risco_ia)
print("IA:", mensagem_ia)

if falhas:
    print("Motivos da abortagem:")
    for falha in falhas:
        print("-", falha)

## Análise energética

Agora calculamos:

- energia disponível antes das perdas;
- energia útil após as perdas;
- consumo estimado da decolagem;
- energia restante após a decolagem;
- autonomia estimada.

Nesta etapa, os valores são obtidos a partir do próprio dataset sintético.

In [ ]:
energia_util, consumo_estimado_decolagem_kwh, energia_restante_apos_decolagem, autonomia_estimada_min = calcular_analise_energetica(
    telemetria["energia_disponivel_kwh"],
    telemetria["perdas_energeticas"],
    telemetria["carga_kw"],
)

print("=== ANÁLISE ENERGÉTICA ===")
print(f"Energia disponível antes das perdas: {telemetria['energia_disponivel_kwh']:.3f} kWh")
print(f"Energia útil após perdas: {energia_util:.3f} kWh")
print(f"Consumo estimado da decolagem (1 min): {consumo_estimado_decolagem_kwh:.3f} kWh")
print(f"Energia restante após decolagem: {energia_restante_apos_decolagem:.3f} kWh")
print(f"Autonomia estimada após perdas: {autonomia_estimada_min:.2f} min")
print(f"Autonomia registrada no dataset: {telemetria['autonomia_estimada_dataset_min']} min")

## Conclusão

O sistema combina duas abordagens:

- **regras fixas:** responsáveis pela decisão final de segurança;
- **IA:** utilizada como apoio para estimativa de risco e detecção de anomalias.

Essa separação continua sendo importante porque, em sistemas críticos, a decisão operacional deve ser baseada em critérios explícitos e auditáveis, enquanto a IA deve atuar como ferramenta complementar de análise.

# Aurora-Singer: Sistema de Telemetria e Verificação de Decolagem

**Projeto Integrador FIAP**

---

## Objetivo
Desenvolver um sistema completo de telemetria para um veículo espacial (Aurora-Singer) capaz de:
- Monitorar parâmetros críticos
- Decidir automaticamente se a decolagem é segura
- Analisar a autonomia energética
- Identificar anomalias com assistência de IA
- Refletir criticamente sobre responsabilidade e sustentabilidade

---
## 1.1 Organização e Descrição da Telemetria

### Estrutura de Dados

O sistema monitora 5 categorias principais de telemetria:

| Parâmetro | Unidade | Faixa Segura | Criticidade |
|-----------|---------|--------------|-------------|
| **Temperatura Interna** | °C | 15-35 | Alta |
| **Temperatura Externa** | °C | -50 a 50 | Alta |
| **Integridade Estrutural** | (0/1) | 1 (íntegro) | Crítica |
| **Nível de Energia** | % | > 85 | Crítica |
| **Pressão dos Tanques** | psi | 1800-2200 | Alta |
| **Status dos Módulos Críticos** | Status | OPERACIONAL | Crítica |

### Definição dos Módulos Críticos
1. **Sistema de Propulsão**: motor principal
2. **Sistema de Comunicação**: transmissão de dados
3. **Sistema de Navegação**: GPS/IMU
4. **Sistema de Segurança**: paraquedas, ejeção de emergência

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import json

# Estrutura de dados de telemetria
telemetria_exemplo = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 28.5,
    'temperatura_externa_celsius': 12.3,
    'integridade_estrutural': 1,  # 1 = íntegro, 0 = danificado
    'nivel_energia_percentual': 92.0,
    'pressao_tanques_psi': 2050,
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'OPERACIONAL',
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

# Exibir estrutura
print("=" * 60)
print("ESTRUTURA DE TELEMETRIA - Aurora-Singer")
print("=" * 60)
print(json.dumps(telemetria_exemplo, indent=2, default=str))

---
## 1.2 Algoritmo de Verificação para Decolagem

### Pseudocódigo

```
FUNÇÃO verificar_decolagem(telemetria):
    
    # Passo 1: Verificar integridade estrutural (gate crítica)
    SE integridade_estrutural == 0:
        RETORNAR "DECOLAGEM ABORTADA" (motivo: estrutura comprometida)
    
    # Passo 2: Verificar status de todos os módulos críticos
    PARA CADA módulo EM módulos_críticos:
        SE status_módulo != "OPERACIONAL":
            RETORNAR "DECOLAGEM ABORTADA" (motivo: módulo inoperante)
    
    # Passo 3: Verificar níveis de energia
    SE nível_energia < 85%:
        RETORNAR "DECOLAGEM ABORTADA" (motivo: energia insuficiente)
    
    # Passo 4: Verificar temperaturas
    SE temperatura_interna < 15 OU temperatura_interna > 35:
        RETORNAR "DECOLAGEM ABORTADA" (motivo: temperatura interna fora da faixa)
    
    SE temperatura_externa < -50 OU temperatura_externa > 50:
        RETORNAR "DECOLAGEM ABORTADA" (motivo: temperatura externa fora da faixa)
    
    # Passo 5: Verificar pressão dos tanques
    SE pressão_tanques < 1800 OU pressão_tanques > 2200:
        RETORNAR "DECOLAGEM ABORTADA" (motivo: pressão dos tanques fora da faixa)
    
    # Passo 6: Todos os testes passaram
    RETORNAR "PRONTO PARA DECOLAR"
FIM
```

### Fluxograma do Algoritmo

```
                    ┌─────────────────┐
                    │ INÍCIO LEITURA  │
                    │  TELEMETRIA     │
                    └────────┬────────┘
                             │
                    ┌────────▼────────┐
                    │ Integridade = 0?├─SIM─► ❌ ABORTADA
                    │                 │       (Estrutura)
                    └────────┬────────┘
                         NÃO │
                    ┌────────▼────────┐
                    │ Módulo = OFF?   ├─SIM─► ❌ ABORTADA
                    │                 │       (Módulo)
                    └────────┬────────┘
                         NÃO │
                    ┌────────▼────────┐
                    │ Energia >= 85%? ├─NÃO─► ❌ ABORTADA
                    │                 │       (Energia)
                    └────────┬────────┘
                        SIM  │
                    ┌────────▼────────┐
                    │ Temp OK?        ├─NÃO─► ❌ ABORTADA
                    │                 │       (Temperatura)
                    └────────┬────────┘
                        SIM  │
                    ┌────────▼────────┐
                    │ Pressão OK?     ├─NÃO─► ❌ ABORTADA
                    │                 │       (Pressão)
                    └────────┬────────┘
                        SIM  │
                    ┌────────▼────────┐
                    │ ✅ PRONTO PARA  │
                    │    DECOLAR      │
                    └─────────────────┘
```

---
## 1.3 Script em Python - Implementação do Algoritmo

### Sistema de Verificação de Decolagem

In [ ]:
class SistemaVerificacaoAurora:
    """Sistema de verificação de segurança para decolagem da Aurora-Singer"""
    
    # Faixas seguras (constantes)
    FAIXA_TEMP_INTERNA = (15, 35)  # °C
    FAIXA_TEMP_EXTERNA = (-50, 50)  # °C
    FAIXA_PRESSAO_TANQUES = (1800, 2200)  # psi
    MINIMO_ENERGIA = 85  # %
    MODULOS_REQUERIDOS = {
        'sistema_propulsao',
        'sistema_comunicacao',
        'sistema_navegacao',
        'sistema_seguranca'
    }
    
    def __init__(self, telemetria):
        self.telemetria = telemetria
        self.resultado = None
        self.motivo = None
        self.detalhes = {}
    
    def verificar_decolagem(self):
        """Executa todas as verificações de segurança"""
        
        # 1. Integridade Estrutural
        if self.telemetria['integridade_estrutural'] == 0:
            self.resultado = "DECOLAGEM ABORTADA"
            self.motivo = "⚠️  Falha na integridade estrutural"
            return self.resultado
        self.detalhes['integridade'] = "✅ Íntegra"
        
        # 2. Módulos Críticos
        for modulo, status in self.telemetria['modulos_criticos'].items():
            if status != 'OPERACIONAL':
                self.resultado = "DECOLAGEM ABORTADA"
                self.motivo = f"⚠️  Módulo {modulo.replace('_', ' ').upper()} inoperante"
                return self.resultado
        self.detalhes['modulos'] = "✅ Todos operacionais"
        
        # 3. Nível de Energia
        if self.telemetria['nivel_energia_percentual'] < self.MINIMO_ENERGIA:
            self.resultado = "DECOLAGEM ABORTADA"
            self.motivo = f"⚠️  Energia insuficiente: {self.telemetria['nivel_energia_percentual']}% (mínimo: {self.MINIMO_ENERGIA}%)"
            return self.resultado
        self.detalhes['energia'] = f"✅ {self.telemetria['nivel_energia_percentual']}%"
        
        # 4. Temperatura Interna
        temp_int = self.telemetria['temperatura_interna_celsius']
        if not (self.FAIXA_TEMP_INTERNA[0] <= temp_int <= self.FAIXA_TEMP_INTERNA[1]):
            self.resultado = "DECOLAGEM ABORTADA"
            self.motivo = f"⚠️  Temperatura interna fora da faixa: {temp_int}°C"
            return self.resultado
        self.detalhes['temp_interna'] = f"✅ {temp_int}°C"
        
        # 5. Temperatura Externa
        temp_ext = self.telemetria['temperatura_externa_celsius']
        if not (self.FAIXA_TEMP_EXTERNA[0] <= temp_ext <= self.FAIXA_TEMP_EXTERNA[1]):
            self.resultado = "DECOLAGEM ABORTADA"
            self.motivo = f"⚠️  Temperatura externa fora da faixa: {temp_ext}°C"
            return self.resultado
        self.detalhes['temp_externa'] = f"✅ {temp_ext}°C"
        
        # 6. Pressão dos Tanques
        pressao = self.telemetria['pressao_tanques_psi']
        if not (self.FAIXA_PRESSAO_TANQUES[0] <= pressao <= self.FAIXA_PRESSAO_TANQUES[1]):
            self.resultado = "DECOLAGEM ABORTADA"
            self.motivo = f"⚠️  Pressão dos tanques fora da faixa: {pressao} psi"
            return self.resultado
        self.detalhes['pressao'] = f"✅ {pressao} psi"
        
        # Todos os testes passaram!
        self.resultado = "PRONTO PARA DECOLAR"
        self.motivo = "✅ Todos os parâmetros dentro dos limites seguros"
        return self.resultado
    
    def gerar_relatorio(self):
        """Gera relatório formatado com resultados"""
        if self.resultado is None:
            self.verificar_decolagem()
        
        print("\n" + "="*70)
        print("RELATÓRIO DE VERIFICAÇÃO - AURORA-SINGER")
        print("="*70)
        print(f"\n🚀 DECISÃO FINAL: {self.resultado}")
        print(f"\n📋 Motivo: {self.motivo}")
        print("\n📊 DETALHES DAS VERIFICAÇÕES:")
        for chave, valor in self.detalhes.items():
            print(f"   • {chave.replace('_', ' ').title()}: {valor}")
        print("\n" + "="*70)

In [ ]:
# TESTE 1: Condições ideais - Deve decolar
print("\n" + "#"*70)
print("# TESTE 1: CONDIÇÕES IDEAIS")
print("#"*70)

telemetria_ideal = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 25.0,
    'temperatura_externa_celsius': 15.0,
    'integridade_estrutural': 1,
    'nivel_energia_percentual': 95.0,
    'pressao_tanques_psi': 2000,
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'OPERACIONAL',
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

verificador1 = SistemaVerificacaoAurora(telemetria_ideal)
verificador1.verificar_decolagem()
verificador1.gerar_relatorio()

In [ ]:
# TESTE 2: Energia insuficiente - Deve abortar
print("\n" + "#"*70)
print("# TESTE 2: ENERGIA INSUFICIENTE")
print("#"*70)

telemetria_sem_energia = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 25.0,
    'temperatura_externa_celsius': 15.0,
    'integridade_estrutural': 1,
    'nivel_energia_percentual': 80.0,  # Abaixo do mínimo de 85%
    'pressao_tanques_psi': 2000,
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'OPERACIONAL',
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

verificador2 = SistemaVerificacaoAurora(telemetria_sem_energia)
verificador2.verificar_decolagem()
verificador2.gerar_relatorio()

In [ ]:
# TESTE 3: Estrutura comprometida - Deve abortar
print("\n" + "#"*70)
print("# TESTE 3: ESTRUTURA COMPROMETIDA")
print("#"*70)

telemetria_danificada = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 25.0,
    'temperatura_externa_celsius': 15.0,
    'integridade_estrutural': 0,  # Danificada
    'nivel_energia_percentual': 95.0,
    'pressao_tanques_psi': 2000,
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'OPERACIONAL',
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

verificador3 = SistemaVerificacaoAurora(telemetria_danificada)
verificador3.verificar_decolagem()
verificador3.gerar_relatorio()

---
## 1.4 Análise Energética

### Cálculo de Autonomia Inicial

In [ ]:
class AnalisadorEnergitico:
    """Analisa autonomia energética da Aurora-Singer"""
    
    def __init__(self, capacidade_kwh, carga_percentual, consumo_decolagem_kw, perda_percentual=0.05):
        """
        Args:
            capacidade_kwh: Capacidade total das baterias (kWh)
            carga_percentual: Carga atual (0-100%)
            consumo_decolagem_kw: Consumo instantâneo na decolagem (kW)
            perda_percentual: Perdas energéticas por calor/eficiência (padrão 5%)
        """
        self.capacidade_kwh = capacidade_kwh
        self.carga_percentual = carga_percentual
        self.consumo_decolagem_kw = consumo_decolagem_kw
        self.perda_percentual = perda_percentual
    
    def calcular_energia_disponivel(self):
        """Calcula energia disponível em kWh"""
        return self.capacidade_kwh * (self.carga_percentual / 100)
    
    def calcular_perdas_energeticas(self):
        """Calcula perdas por calor e ineficiência"""
        energia_disponivel = self.calcular_energia_disponivel()
        perdas = energia_disponivel * self.perda_percentual
        return perdas
    
    def calcular_energia_util(self):
        """Calcula energia realmente utilizável"""
        energia_disponivel = self.calcular_energia_disponivel()
        perdas = self.calcular_perdas_energeticas()
        return energia_disponivel - perdas
    
    def calcular_tempo_autonomia_horas(self):
        """Calcula tempo de autonomia em horas"""
        energia_util = self.calcular_energia_util()
        tempo_horas = energia_util / self.consumo_decolagem_kw
        return tempo_horas
    
    def calcular_tempo_autonomia_minutos(self):
        """Calcula tempo de autonomia em minutos"""
        return self.calcular_tempo_autonomia_horas() * 60
    
    def gerar_relatorio_energetico(self):
        """Gera relatório completo de energia"""
        print("\n" + "="*70)
        print("ANÁLISE ENERGÉTICA - AURORA-SINGER")
        print("="*70)
        
        print(f"\n📊 PARÂMETROS:")
        print(f"   • Capacidade total das baterias: {self.capacidade_kwh} kWh")
        print(f"   • Carga atual: {self.carga_percentual}%")
        print(f"   • Consumo na decolagem: {self.consumo_decolagem_kw} kW")
        print(f"   • Perda energética estimada: {self.perda_percentual*100}%")
        
        energia_disp = self.calcular_energia_disponivel()
        perdas = self.calcular_perdas_energeticas()
        energia_util = self.calcular_energia_util()
        tempo_horas = self.calcular_tempo_autonomia_horas()
        tempo_minutos = self.calcular_tempo_autonomia_minutos()
        
        print(f"\n⚡ CÁLCULOS:")
        print(f"   • Energia disponível: {energia_disp:.2f} kWh")
        print(f"   • Perdas energéticas: {perdas:.2f} kWh ({self.perda_percentual*100}%)")
        print(f"   • Energia útil: {energia_util:.2f} kWh")
        
        print(f"\n🕐 AUTONOMIA:")
        print(f"   • Tempo de voo estimado: {tempo_horas:.2f} horas")
        print(f"   • Tempo de voo estimado: {tempo_minutos:.0f} minutos")
        
        # Aviso se autonomia for curta
        if tempo_minutos < 30:
            print(f"\n⚠️  AVISO: Autonomia crítica! Menos de 30 minutos de voo.")
        elif tempo_minutos < 60:
            print(f"\n⚠️  ATENÇÃO: Autonomia reduzida. Menos de 1 hora de voo.")
        else:
            print(f"\n✅ Autonomia adequada para missão.")
        
        print("\n" + "="*70)

In [ ]:
# Simulação de análise energética
# Aurora-Singer: Capacidade 150 kWh, Carga 95%, Consumo 85 kW na decolagem

analisador = AnalisadorEnergitico(
    capacidade_kwh=150,
    carga_percentual=95,
    consumo_decolagem_kw=85,
    perda_percentual=0.05
)

analisador.gerar_relatorio_energetico()

# Simulação adicional com carga menor
print("\n\n" + "#"*70)
print("# SIMULAÇÃO 2: Carga Reduzida (70%)")
print("#"*70)

analisador2 = AnalisadorEnergitico(
    capacidade_kwh=150,
    carga_percentual=70,
    consumo_decolagem_kw=85,
    perda_percentual=0.05
)

analisador2.gerar_relatorio_energetico()

---
## 1.5 Análise Assistida por IA

### Classificação, Anomalias e Sugestões de Risco

In [ ]:
class AnalisadorIAAurora:
    """Análise assistida por IA para detecção de anomalias e risco"""
    
    # Rangos normais de operação (baseados em histórico de voos bem-sucedidos)
    RANGOS_NORMAIS = {
        'temperatura_interna_celsius': (20, 30),  # Range confortável
        'temperatura_externa_celsius': (-30, 40),  # Range expandido
        'nivel_energia_percentual': (90, 100),  # Deve estar no topo
        'pressao_tanques_psi': (1950, 2050)  # Range estreito
    }
    
    LIMIARES_ANOMALIA = {
        'temperatura_interna_extrema': (10, 40),
        'mudanca_temp_rapida': 5,  # °C por leitura
        'flutuacao_pressao': 50  # psi de oscilação
    }
    
    def __init__(self, telemetria, telemetria_anterior=None):
        self.telemetria = telemetria
        self.telemetria_anterior = telemetria_anterior
        self.anomalias = []
        self.riscos = []
        self.recomendacoes = []
    
    def classificar_dados(self):
        """Classifica cada parâmetro como NORMAL, PRECAUÇÃO ou CRÍTICO"""
        classificacao = {}
        
        # Classificar temperatura interna
        temp_int = self.telemetria['temperatura_interna_celsius']
        if self.RANGOS_NORMAIS['temperatura_interna_celsius'][0] <= temp_int <= self.RANGOS_NORMAIS['temperatura_interna_celsius'][1]:
            classificacao['temperatura_interna'] = 'NORMAL'
        elif 15 <= temp_int <= 35:
            classificacao['temperatura_interna'] = 'PRECAUÇÃO'
        else:
            classificacao['temperatura_interna'] = 'CRÍTICO'
        
        # Classificar energia
        energia = self.telemetria['nivel_energia_percentual']
        if energia >= 90:
            classificacao['energia'] = 'NORMAL'
        elif energia >= 85:
            classificacao['energia'] = 'PRECAUÇÃO'
        else:
            classificacao['energia'] = 'CRÍTICO'
        
        # Classificar pressão
        pressao = self.telemetria['pressao_tanques_psi']
        if 1950 <= pressao <= 2050:
            classificacao['pressao'] = 'NORMAL'
        elif 1800 <= pressao <= 2200:
            classificacao['pressao'] = 'PRECAUÇÃO'
        else:
            classificacao['pressao'] = 'CRÍTICO'
        
        # Classificação de módulos
        status_modulos = all(s == 'OPERACIONAL' for s in self.telemetria['modulos_criticos'].values())
        classificacao['modulos'] = 'NORMAL' if status_modulos else 'CRÍTICO'
        
        return classificacao
    
    def detectar_anomalias(self):
        """Identifica possíveis anomalias nos dados"""
        self.anomalias = []
        
        # Anomalia 1: Temperatura interna muito baixa ou alta
        temp_int = self.telemetria['temperatura_interna_celsius']
        if temp_int < self.LIMIARES_ANOMALIA['temperatura_interna_extrema'][0]:
            self.anomalias.append({
                'tipo': 'Temperatura Interna Baixa',
                'valor': f"{temp_int}°C",
                'severidade': 'ALTA'
            })
        elif temp_int > self.LIMIARES_ANOMALIA['temperatura_interna_extrema'][1]:
            self.anomalias.append({
                'tipo': 'Temperatura Interna Elevada',
                'valor': f"{temp_int}°C",
                'severidade': 'ALTA'
            })
        
        # Anomalia 2: Mudança rápida de temperatura
        if self.telemetria_anterior:
            delta_temp = abs(temp_int - self.telemetria_anterior['temperatura_interna_celsius'])
            if delta_temp > self.LIMIARES_ANOMALIA['mudanca_temp_rapida']:
                self.anomalias.append({
                    'tipo': 'Mudança Rápida de Temperatura',
                    'valor': f"{delta_temp}°C",
                    'severidade': 'MÉDIA'
                })
        
        # Anomalia 3: Pressão instável
        pressao = self.telemetria['pressao_tanques_psi']
        if pressao < 1800 or pressao > 2200:
            self.anomalias.append({
                'tipo': 'Pressão de Tanque Anômala',
                'valor': f"{pressao} psi",
                'severidade': 'ALTA'
            })
        
        # Anomalia 4: Integridade comprometida
        if self.telemetria['integridade_estrutural'] == 0:
            self.anomalias.append({
                'tipo': 'Falha Crítica de Integridade',
                'valor': 'Estrutura comprometida',
                'severidade': 'CRÍTICA'
            })
        
        return self.anomalias
    
    def analisar_risco(self):
        """Avalia nível de risco geral da missão"""
        self.riscos = []
        nivel_risco_total = 0
        
        classificacao = self.classificar_dados()
        anomalias = self.detectar_anomalias()
        
        # Score de risco por anomalia
        if any(a['severidade'] == 'CRÍTICA' for a in anomalias):
            nivel_risco_total += 4
            self.riscos.append("Presença de anomalia CRÍTICA detectada")
        
        if any(a['severidade'] == 'ALTA' for a in anomalias):
            nivel_risco_total += 2
            self.riscos.append("Múltiplas anomalias com severidade ALTA")
        
        if classificacao['energia'] == 'CRÍTICO':
            nivel_risco_total += 2
            self.riscos.append("Energia em nível crítico")
        
        if classificacao['temperatura_interna'] == 'CRÍTICO':
            nivel_risco_total += 2
            self.riscos.append("Temperatura interna fora dos limites")
        
        if classificacao['modulos'] == 'CRÍTICO':
            nivel_risco_total += 3
            self.riscos.append("Um ou mais módulos críticos inoperantes")
        
        return min(nivel_risco_total, 10)  # Score 0-10
    
    def gerar_recomendacoes(self):
        """Gera sugestões baseadas na análise"""
        self.recomendacoes = []
        classificacao = self.classificar_dados()
        
        if classificacao['energia'] == 'PRÉCAUÇÃO':
            self.recomendacoes.append("⚡ Recarregar baterias antes da próxima missão")
        
        if classificacao['temperatura_interna'] == 'PRECAUÇÃO':
            self.recomendacoes.append("❄️  Ativar sistemas de refrigeração se temperatura continuar subindo")
        
        if classificacao['pressao'] == 'PRECAUÇÃO':
            self.recomendacoes.append("🛠️  Inspecionar válvulas de pressão dos tanques")
        
        if self.telemetria['nivel_energia_percentual'] < 85:
            self.recomendacoes.append("⚠️  NÃO DECOLAR: Energia abaixo do mínimo seguro")
        
        if not self.recomendacoes:
            self.recomendacoes.append("✅ Sistema totalmente operacional. Pronto para missão.")
        
        return self.recomendacoes
    
    def gerar_relatorio_ia(self):
        """Gera relatório completo de análise de IA"""
        print("\n" + "="*70)
        print("ANÁLISE ASSISTIDA POR IA - AURORA-SINGER")
        print("="*70)
        
        # Classificações
        classificacao = self.classificar_dados()
        print(f"\n📊 CLASSIFICAÇÃO DE PARÂMETROS:")
        for param, status in classificacao.items():
            emoji = "✅" if status == "NORMAL" else "⚠️" if status == "PRECAUÇÃO" else "🔴"
            print(f"   {emoji} {param.replace('_', ' ').title()}: {status}")
        
        # Anomalias
        anomalias = self.detectar_anomalias()
        print(f"\n🔍 ANOMALIAS DETECTADAS:")
        if anomalias:
            for anomalia in anomalias:
                emoji = "🔴" if anomalia['severidade'] == 'CRÍTICA' else "🟠" if anomalia['severidade'] == 'ALTA' else "🟡"
                print(f"   {emoji} {anomalia['tipo']}: {anomalia['valor']} ({anomalia['severidade']})")
        else:
            print("   ✅ Nenhuma anomalia detectada")
        
        # Análise de Risco
        nivel_risco = self.analisar_risco()
        print(f"\n📈 ANÁLISE DE RISCO:")
        print(f"   Nível de Risco Total: {nivel_risco}/10", end="")
        if nivel_risco <= 2:
            print(" ✅ (Baixo)")
        elif nivel_risco <= 5:
            print(" ⚠️  (Médio)")
        else:
            print(" 🔴 (Alto)")
        
        if self.riscos:
            print("\n   Fatores de Risco:")
            for risco in self.riscos:
                print(f"   • {risco}")
        
        # Recomendações
        print(f"\n💡 RECOMENDAÇÕES:")
        recomendacoes = self.gerar_recomendacoes()
        for rec in recomendacoes:
            print(f"   {rec}")
        
        print("\n" + "="*70)

In [ ]:
# Análise de IA - Cenário 1: Sistema Normal
telemetria_normal = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 24.0,
    'temperatura_externa_celsius': 15.0,
    'integridade_estrutural': 1,
    'nivel_energia_percentual': 95.0,
    'pressao_tanques_psi': 2000,
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'OPERACIONAL',
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

analisador_ia1 = AnalisadorIAAurora(telemetria_normal)
analisador_ia1.gerar_relatorio_ia()

In [ ]:
# Análise de IA - Cenário 2: Anomalias Detectadas
telemetria_anomala = {
    'timestamp': datetime.now(),
    'temperatura_interna_celsius': 38.5,  # Elevada
    'temperatura_externa_celsius': 45.0,  # Muito quente
    'integridade_estrutural': 1,
    'nivel_energia_percentual': 87.0,  # Abaixo do ideal
    'pressao_tanques_psi': 2150,  # Um pouco elevada
    'modulos_criticos': {
        'sistema_propulsao': 'OPERACIONAL',
        'sistema_comunicacao': 'FALHA',  # Módulo com falha!
        'sistema_navegacao': 'OPERACIONAL',
        'sistema_seguranca': 'OPERACIONAL'
    }
}

analisador_ia2 = AnalisadorIAAurora(telemetria_anomala)
analisador_ia2.gerar_relatorio_ia()

---
## 1.6 Reflexão Crítica

### Ética, Responsabilidade e Impacto Social da Exploração Espacial

#### 1. Ética e Responsabilidade

A exploração espacial, independentemente de seu objetivo (comercial, científico ou militar), raise profound ethical questions:

**Responsabilidade com Vidas Humanas:**
- Sistemas como o Aurora-Singer são projetos complexos onde a falha pode resultar em perda de vidas
- A implementação rigorosa de algoritmos de verificação (como o apresentado neste projeto) é uma responsabilidade moral
- O teste exaustivo e validação de sistemas de segurança não é apenas uma medida técnica, mas um imperativo ético

**Equidade e Acesso:**
- Atualmente, a exploração espacial é dominada por países desenvolvidos e corporações ricas
- Questiona-se se os benefícios da tecnologia espacial chegam de forma equitativa à população global
- É imperativo garantir que avanços espaciais beneficiem a humanidade como um todo, não apenas elites

**Governança e Regulação:**
- Falta de marcos regulatórios claros para atividades espaciais comerciais
- Necessidade de acordos internacionais que garantam responsabilidade e segurança
- Importância de transparência em programas espaciais militares

#### 2. Impacto Social da Exploração Espacial

**Benefícios Potenciais:**
- **Tecnologia transferida**: Inovações espaciais alimentaram avanços em comunicação, meteorologia, medicina
- **Conhecimento científico**: Compreensão da origem do universo, busca por vida extraterrestre
- **Recursos futuro**: Potencial de mineração espacial e colonização para garantir sobrevivência humana
- **Inspiração**: A exploração espacial inspira gerações a perseguir carreiras STEM

**Desafios e Riscos:**
- **Custo de oportunidade**: Bilhões gastos em exploração espacial poderiam resolver problemas terrestres urgentes (fome, pobreza, saúde)
- **Poluição ambiental**: Lançamentos frequentes contribuem para emissões de carbono e poluição atmosférica
- **Lixo espacial**: Detritos orbitais representam risco crescente para futuras missões
- **Militarização**: Risco de armas baseadas no espaço exacerbando conflitos globais
- **Colonialismo espacial**: Possibilidade de exploração de recursos extraterrestres por potências dominantes

#### 3. Sustentabilidade Tecnológica

**Desafios de Sustentabilidade:**
- **Consumo de recursos**: Foguetes reutilizáveis reduzem impacto, mas ainda requerem combustíveis fósseis
- **Lixo espacial**: Necessidade urgente de tecnologias para remover detritos, evitando efeito cascata (Síndrome de Kessler)
- **Responsabilidade corporativa**: Companhias privadas de exploração espacial devem manter padrões ambientais rigorosos
- **Pesquisa de combustíveis limpos**: Desenvolvimento de combustíveis sustentáveis para propulsão espacial

**Caminho para Sustentabilidade:**
1. **Regulação ambiental rigorosa** para lançamentos espaciais
2. **Investimento em tecnologias limpas** de propulsão
3. **Recuperação de lixo espacial** como prioridade
4. **Compartilhamento internacional de dados** e infraestrutura espacial
5. **Integração com objetivos de desenvolvimento sustentável** da ONU
6. **Educação pública** sobre impacto ambiental da exploração espacial

#### 4. Conclusão

A exploração espacial é um empreendimento intrinsecamente humano que reflete nossos valores e aspirações. Contudo, deve ser conduzida com responsabilidade ética, considerando impactos sociais e ambientais. Projetos como o Aurora-Singer representam o potencial tecnológico humano, mas este potencial deve ser exercido com sabedoria e responsabilidade para garantir que beneficie a humanidade no presente e no futuro.

**"O espaço é a última fronteira, mas não devemos deixar a Terra para trás."**

---

## Resumo Executivo

### Requisitos Atendidos:

✅ **1.1 - Organização e Descrição da Telemetria**: Estrutura completa com 6 parâmetros monitorados

✅ **1.2 - Algoritmo de Verificação**: Pseudocódigo detalhado + fluxograma visual

✅ **1.3 - Script em Python**: Implementação completa com 3 cenários de teste

✅ **1.4 - Análise Energética**: Cálculos de autonomia com 2 simulações

✅ **1.5 - Análise Assistida por IA**: Classificação, anomalias e riscos com 2 cenários

✅ **1.6 - Reflexão Crítica**: Discussão abrangente sobre ética, impacto social e sustentabilidade

### Estatísticas do Projeto:
- **Linhas de Código**: ~400
- **Classes Implementadas**: 3 (Verificação, Análise Energética, Análise IA)
- **Cenários Testados**: 5
- **Parâmetros Monitorados**: 6
- **Pontos de Verificação**: 6